[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Red1-Rahman/NiriZan/blob/main/experiments/01_instrumentation_trace_storage.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https://github.com/Red1-Rahman/NiriZan/blob/main/experiments/01_instrumentation_trace_storage.ipynb)

# Experiment 01: Instrumentation & Trace Storage
**Phase 1 Exploration**: Validating asynchronous span emission, immutable trace creation, schema constraints, and storage repository interfaces under simulated application loads.

## 1. Environment Setup
Install `nirizan` directly from the main branch or setup local dev dependencies.

In [1]:
import sys

# Install NiriZan and dependencies silently when running in Colab/Kaggle
!pip install -q pydantic>=2.7 "git+https://github.com/Red1-Rahman/NiriZan.git@main#egg=nirizan"

from datetime import datetime, timezone
import time
from uuid import uuid4
import asyncio

from nirizan.instrumentation.spans import Span, SpanKind, Trace
from pydantic import ValidationError

print("✅ NiriZan successfully loaded!")

✅ NiriZan successfully loaded!


## 2. Validating `Span` Model Contracts
Spans are the atomic unit of instrumentation[cite: 6]. We test:
1. Span creation across all `SpanKind` enums (`PLANNING`, `RETRIEVAL`, `TOOL_USE`, `GENERATION`)[cite: 6].
2. Strict frozen immutability (`frozen=True`)[cite: 6].
3. Attribute primitive typing constraints[cite: 6].

In [2]:
trace_id = uuid4()
now = datetime.now(timezone.utc)

# 1. Create a retrieval span
retrieval_span = Span(
    span_id=uuid4(),
    trace_id=trace_id,
    kind=SpanKind.RETRIEVAL,
    name="qdrant_vector_search",
    started_at=now,
    ended_at=now,
    attributes={"top_k": 5, "vector_dim": 1536, "hybrid_search": True},
    input_payload="What is continuous evaluation?",
    output_payload="Doc 1: Continuous evaluation infrastructure...",
)

print(f"Created Span ID: {retrieval_span.span_id}")
print(f"Attributes: {retrieval_span.attributes}")

# 2. Test Immutability
try:
    retrieval_span.name = "modified_name"  # type: ignore
except ValidationError as e:
    print("\n✅ Immutability verified: Cannot modify frozen Span instance!")

Created Span ID: a034db1a-5ca5-43ca-9afb-4cd766d0146e
Attributes: {'top_k': 5, 'vector_dim': 1536, 'hybrid_search': True}

✅ Immutability verified: Cannot modify frozen Span instance!


## 3. Assembling a `Trace`
A `Trace` is an ordered collection of spans belonging to a single application invocation[cite: 6].
We test:
1. `trace_id` validation across child spans[cite: 6].
2. Filtering spans by `SpanKind` via `spans_of_kind()`[cite: 6].

In [3]:
generation_span = Span(
    span_id=uuid4(),
    trace_id=trace_id,
    kind=SpanKind.GENERATION,
    name="llm_generate_answer",
    started_at=now,
    ended_at=now,
    attributes={"model": "gpt-4o", "temperature": 0.2},
    input_payload="Context: ... Prompt: What is continuous evaluation?",
    output_payload="Continuous evaluation is an engineering layer...",
)

# Create Trace
trace = Trace(
    trace_id=trace_id,
    application_name="production_rag_service",
    spans=[retrieval_span, generation_span],
    created_at=now,
)

print(f"Trace ID: {trace.trace_id}")
print(f"Total Spans: {len(trace.spans)}")
print(f"Retrieval Spans: {len(trace.spans_of_kind(SpanKind.RETRIEVAL))}")
print(f"Generation Spans: {len(trace.spans_of_kind(SpanKind.GENERATION))}")

# Test trace_id mismatch assertion
mismatched_span = Span(
    span_id=uuid4(),
    trace_id=uuid4(),  # Different trace_id!
    kind=SpanKind.PLANNING,
    name="query_planner",
    started_at=now,
    ended_at=now,
)

try:
    Trace(
        trace_id=trace_id,
        application_name="invalid_trace_app",
        spans=[mismatched_span],
        created_at=now,
    )
except ValidationError:
    print("\n✅ Trace ID validation verified: Rejected mismatched span!")

Trace ID: a6ea2b3a-6daa-4913-aff0-bb3d35dbfa6e
Total Spans: 2
Retrieval Spans: 1
Generation Spans: 1

✅ Trace ID validation verified: Rejected mismatched span!


## 4. Measuring Tracing Overhead (Latency Benchmark)
Instrumentation must **never** block the application's request/response path[cite: 6].
We simulate async background trace exportation to verify minimal latency impact.

In [4]:
class MockAsyncExporter:
    """Simulates an asynchronous background collector export."""

    async def export(self, trace: Trace) -> None:
        # Simulate background network/storage latency without blocking caller
        await asyncio.sleep(0.05)


async def simulate_application_request(exporter: MockAsyncExporter):
    start_time = time.perf_counter()

    # Application execution simulation
    t_id = uuid4()
    n = datetime.now(timezone.utc)
    s = Span(
        span_id=uuid4(),
        trace_id=t_id,
        kind=SpanKind.GENERATION,
        name="rag_response",
        started_at=n,
        ended_at=n,
    )
    tr = Trace(
        trace_id=t_id, application_name="benchmark_app", spans=[s], created_at=n
    )

    # Fire-and-forget background task for trace emission
    asyncio.create_task(exporter.export(tr))

    elapsed_ms = (time.perf_counter() - start_time) * 1000
    return elapsed_ms


async def run_benchmark():
    exporter = MockAsyncExporter()
    latencies = []

    for _ in range(1000):
        latency = await simulate_application_request(exporter)
        latencies.append(latency)

    avg_latency = sum(latencies) / len(latencies)
    print(f"⚡ Tracing Latency Overhead across 1,000 runs:")
    print(f"   Average Overhead: {avg_latency:.4f} ms per request")
    print(
        f"   Max Single-Request Overhead: {max(latencies):.4f} ms (Non-blocking verified)"
    )


await run_benchmark()

⚡ Tracing Latency Overhead across 1,000 runs:
   Average Overhead: 0.0237 ms per request
   Max Single-Request Overhead: 1.3122 ms (Non-blocking verified)


Emitting traces via fire-and-forget background tasks imposes practically zero latency penalty on the caller application path (averaging $\approx 0.015\text{ ms}$ per request). The non-blocking constraint is verified.



---



## 5. Prototyping Context-Aware `Tracer` (`contextvars`)
Manual instantiation of UUIDs and timestamps is error-prone. A `Tracer` uses Python's built-in `contextvars` module to track the active `trace_id` and `current_span_id` across nested function calls automatically without requiring explicit argument passing.

We test:
1. Implicit context propagation across nested execution contexts.
2. Automatic `parent_span_id` linking between parent and child spans.

In [8]:
import contextvars
from contextlib import asynccontextmanager
from datetime import datetime, timezone
import functools
from typing import Any, AsyncGenerator, Callable, Optional
from uuid import UUID, uuid4

# Context variables for thread/async-safe execution tracing
_current_trace_id: contextvars.ContextVar[Optional[UUID]] = (
    contextvars.ContextVar("current_trace_id", default=None)
)
_current_span_id: contextvars.ContextVar[Optional[UUID]] = (
    contextvars.ContextVar("current_span_id", default=None)
)


class PrototypeTracer:
    """Manages span lifecycle and context propagation."""

    def __init__(self, application_name: str = "demo_app"):
        self.application_name = application_name
        self.active_spans: list[Span] = []

    @asynccontextmanager
    async def start_span(
        self,
        name: str,
        kind: SpanKind,
        attributes: dict[str, Any] | None = None,
        input_payload: str | None = None,
    ) -> AsyncGenerator[UUID, None]:
        # 1. Resolve or initialize Trace ID
        trace_id = _current_trace_id.get()
        if trace_id is None:
            trace_id = uuid4()
            _current_trace_id.set(trace_id)

        # 2. Automatically capture parent span ID from current context
        parent_span_id = _current_span_id.get()

        # 3. Create new span & set start timestamp
        span_id = uuid4()
        started_at = datetime.now(timezone.utc)

        # Set this span as the current active span context
        token_span = _current_span_id.set(span_id)

        try:
            yield span_id
        finally:
            ended_at = datetime.now(timezone.utc)

            # Create immutable span record on exit
            completed_span = Span(
                span_id=span_id,
                trace_id=trace_id,
                parent_span_id=parent_span_id,
                kind=kind,
                name=name,
                started_at=started_at,
                ended_at=ended_at,
                attributes=attributes or {},
                input_payload=input_payload,
            )
            self.active_spans.append(completed_span)

            # Restore previous span context
            _current_span_id.reset(token_span)

    def get_assembled_trace(self) -> Trace:  # Added `self` parameter
        trace_id = _current_trace_id.get()
        return Trace(
            trace_id=trace_id or uuid4(),
            application_name=self.application_name,
            spans=list(self.active_spans),
            created_at=datetime.now(timezone.utc),
        )


print("✅ PrototypeTracer with contextvars initialized successfully!")

✅ PrototypeTracer with contextvars initialized successfully!


## 6. Prototyping Developer SDK (`@trace_span` Decorator)
To keep application code clean, we prototype a decorator SDK wrapper around `Tracer`.

We test:
1. End-to-end tracing of a nested RAG pipeline execution tree.
2. Verification of implicit parent-child relationship (`root` -> `retrieval` + `generation`).

In [9]:
tracer = PrototypeTracer(application_name="rag_sdk_experiment")


def trace_span(kind: SpanKind, name: str | None = None):
    """Decorator to automatically instrument async functions."""

    def decorator(func: Callable):
        span_name = name or func.__name__

        @functools.wraps(func)
        async def wrapper(*args, **kwargs):
            input_str = str(args[0]) if args else str(kwargs)
            async with tracer.start_span(
                name=span_name, kind=kind, input_payload=input_str
            ):
                return await func(*args, **kwargs)

        return wrapper

    return decorator


# --- Instrumented Application Services ---


@trace_span(kind=SpanKind.RETRIEVAL, name="qdrant_vector_fetch")
async def fetch_documents(query: str) -> list[str]:
    await asyncio.sleep(0.01)  # Simulate DB latency
    return ["Doc 1: Continuous evaluation...", "Doc 2: Telemetry instrumentation..."]


@trace_span(kind=SpanKind.GENERATION, name="openai_llm_call")
async def generate_response(query: str, docs: list[str]) -> str:
    await asyncio.sleep(0.02)  # Simulate LLM inference latency
    return "Continuous evaluation infrastructure tracks real-time performance."


@trace_span(kind=SpanKind.PLANNING, name="rag_orchestrator")
async def run_rag_pipeline(user_query: str) -> str:
    docs = await fetch_documents(user_query)
    answer = await generate_response(user_query, docs)
    return answer


# --- Run Experiment ---


async def run_sdk_verification():
    answer = await run_rag_pipeline("What is continuous evaluation?")
    assembled_trace = tracer.get_assembled_trace()

    print(f"Pipeline Result: '{answer}'\n")
    print(f"📊 Trace Verification (Trace ID: {assembled_trace.trace_id}):")
    print(f"   Total Spans Captured: {len(assembled_trace.spans)}")

    # Identify Root Span vs Child Spans
    root_spans = [s for s in assembled_trace.spans if s.parent_span_id is None]
    child_spans = [
        s for s in assembled_trace.spans if s.parent_span_id is not None
    ]

    print(f"   Root Spans: {len(root_spans)} ({root_spans[0].name})")
    print(f"   Child Spans: {len(child_spans)}")

    # Assertions
    root_span = root_spans[0]
    for child in child_spans:
        assert (
            child.parent_span_id == root_span.span_id
        ), f"Child span {child.name} parent_span_id mismatch!"
        assert (
            child.trace_id == root_span.trace_id
        ), f"Child span {child.name} trace_id mismatch!"
        print(
            f"   └─ Span '{child.name}' ({child.kind.value}) correctly linked to Parent '{root_span.name}'"
        )

    print(
        "\n✅ Automatic context propagation & SDK parent-child linking verified!"
    )


await run_sdk_verification()

Pipeline Result: 'Continuous evaluation infrastructure tracks real-time performance.'

📊 Trace Verification (Trace ID: b347dd9e-f4ac-4312-a65d-67003e1830d8):
   Total Spans Captured: 3
   Root Spans: 1 (rag_orchestrator)
   Child Spans: 2
   └─ Span 'qdrant_vector_fetch' (retrieval) correctly linked to Parent 'rag_orchestrator'
   └─ Span 'openai_llm_call' (generation) correctly linked to Parent 'rag_orchestrator'

✅ Automatic context propagation & SDK parent-child linking verified!
